# 発展課題2（演習2：スレッドセーフなキュー）

3問あります。**全部やる必要はありません。**

- [問2-1 `size()` を見てから `push` してよいか](#scrollTo=adv02_queue_01)（実験）
- [問2-2 条件変数を1本にまとめたら](#scrollTo=adv02_queue_05)（実験・キューの中身の話）
- [問2-3 待つ `pop` と待たない `pop`](#scrollTo=adv02_queue_09)（考察）

## 問2-1. `size()` を見てから `push` してよいか

キューに3個までしか入れたくない、と思ったとします。こう書けばよいでしょうか。

```cpp
if (q.size() < 3) q.push(x);
```

`size()` も `push()` も、それぞれは鍵で正しく守られています。**期待どおりに動くでしょうか。**
作る係は4人います。

In [ ]:
%%writefile adv02a.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <atomic>
#include <vector>
using namespace std;

// push も size も、それぞれは鍵で正しく守られている
class SafeQueue {
public:
    void push(int v) {
        lock_guard<mutex> g(mtx_);
        q_.push(v);
        if (q_.size() > peak_) peak_ = q_.size();
    }
    bool pop(int& out) {
        lock_guard<mutex> g(mtx_);
        if (q_.empty()) return false;
        out = q_.front(); q_.pop();
        return true;
    }
    size_t size() const { lock_guard<mutex> g(mtx_); return q_.size(); }
    size_t peak() const { lock_guard<mutex> g(mtx_); return peak_; }
private:
    queue<int> q_;
    size_t peak_ = 0;
    mutable mutex mtx_;
};

SafeQueue q;
atomic<bool> stop{false};

// 「3個未満なら入れる」を、使う側で書いた場合
void producer() {
    for (int i = 0; i < 200000; i++) {
        if (q.size() < 3) q.push(i);        // ← 確かめてから入れる
    }
}

void consumer() {
    int v;
    while (!stop) q.pop(v);
}

int main() {
    thread c(consumer);
    vector<thread> ps;
    for (int k = 0; k < 4; k++) ps.emplace_back(producer);   // 作る係4人
    for (auto& t : ps) t.join();
    stop = true; c.join();
    cout << "「3個未満なら入れる」と書いたのに、実際に並んだ最大の個数 = "
         << q.peak() << " 個\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread adv02a.cpp -o adv02a
!for i in 1 2 3; do ./adv02a; done

### 問2-1 の解答 ―― 動きません。3個を超えます

```
「3個未満なら入れる」と書いたのに、実際に並んだ最大の個数 = 5 個
```

問題は `size()` でも `push()` でもなく、**その2つの「あいだ」**です。

```cpp
if (q.size() < 3)      // ← ここで鍵をかけ、答えを受け取り、鍵を開ける
                       // ★ この隙間に、他のスレッドが push できてしまう
    q.push(x);         // ← ここでまた鍵をかける
```

`size()` が返した「2個」という答えは、**返した瞬間から過去の情報**です。
作る係が4人いれば、4人とも「いま2個だ」と見てから、4人とも入れます。

演習2-1 の②とまったく同じ罠です。

> **「確かめてから使う」は壊れる。確かめることと使うことを、同じ鍵の中でやらなければならない。**

`ConcurrentQueue` の `push` を見てください。

```cpp
std::unique_lock<std::mutex> lk(mtx_);
can_push_.wait(lk, [this] { return q_.size() < capacity_; });   // 確かめる
q_.push(v);                                                     // 入れる
```

**確かめるのと入れるのが、1本の鍵の中でつながっています。** 隙間がないので容量が守られます。
だから容量は**コンストラクタで渡す**（`ConcurrentQueue<int> q(3);`）のであって、
使う側が `size()` で管理するのではありません。

> `size()` は「いまどれくらい混んでいるか」を**表示・記録する**のには使えます。
> 使ってはいけないのは、**その値をもとに判断して動く**ことです（付録・発展課題3-3 は前者の使い方）。

## 問2-2. 条件変数を1本にまとめたら

`ConcurrentQueue` は条件変数を2本持っています。

- `can_pop_` … 「取り出せるようになった」（空でなくなった）
- `can_push_` … 「入れられるようになった」（満杯でなくなった）

**これを1本にまとめたら、どうなるでしょうか。**
`notify_one()` の場合と `notify_all()` の場合で答えは変わるでしょうか。

（容量1・作る係2人・受け取る係1人という、待ち人の種類が混ざる状況で試します。
最後の1本は止まるので5秒で強制終了します。**終了コード 124 が「止まった」印**です）

In [ ]:
%%writefile adv02b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
using namespace std;

enum Mode { TWO_CV, ONE_CV_ALL, ONE_CV_ONE };

template <typename T>
class Q {
public:
    Q(size_t cap, Mode m) : capacity_(cap), mode_(m) {}

    void push(const T& v) {
        unique_lock<mutex> lk(mtx_);
        cv_push().wait(lk, [this] { woke_++; return q_.size() < capacity_; });
        q_.push(v);
        lk.unlock();
        wake(cv_pop());
    }
    T pop() {
        unique_lock<mutex> lk(mtx_);
        cv_pop().wait(lk, [this] { woke_++; return !q_.empty(); });
        T v = q_.front(); q_.pop();
        lk.unlock();
        wake(cv_push());
        return v;
    }
    long woke() const { return woke_; }

private:
    // 2本モードでは別々の条件変数、1本モードでは同じものを返す
    condition_variable& cv_pop()  { return a_; }
    condition_variable& cv_push() { return (mode_ == TWO_CV) ? b_ : a_; }
    void wake(condition_variable& cv) {
        if (mode_ == ONE_CV_ALL) cv.notify_all(); else cv.notify_one();
    }
    queue<T> q_;
    size_t capacity_;
    Mode mode_;
    long woke_ = 0;
    mutex mtx_;
    condition_variable a_, b_;
};

void run(Mode m, const char* label) {
    Q<int> q(1, m);                       // 容量1、作る係2人、受け取る係1人
    cout << label << flush;
    vector<thread> ts;
    ts.emplace_back([&] { for (int i = 0; i < 10; i++) q.push(i); });
    ts.emplace_back([&] { for (int i = 0; i < 10; i++) q.push(i); });
    ts.emplace_back([&] { for (int i = 0; i < 20; i++) q.pop(); });
    for (auto& t : ts) t.join();
    cout << " 20個すべて処理できた。述語を評価した回数 = " << q.woke() << "\n" << flush;
}

int main() {
    run(TWO_CV,     "【条件変数2本 + notify_one（正しい形）】");
    run(ONE_CV_ALL, "【条件変数1本 + notify_all      】");
    run(ONE_CV_ONE, "【条件変数1本 + notify_one      】");
    cout << "ここには到達しない\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread adv02b.cpp -o adv02b
!timeout 5 ./adv02b; echo "終了コード=$? （124 なら止まった）"

### 問2-2 の解答 ―― 1本 + `notify_one` は止まる

```
【条件変数2本 + notify_one（正しい形）】 20個すべて処理できた
【条件変数1本 + notify_all      】 20個すべて処理できた
【条件変数1本 + notify_one      】 ← ここで止まる（終了コード 124）
```

条件変数が1本だと、そこに**種類の違う待ち人が混ざります。**

- 「空でなくなるのを待っている人」（取り出したい）
- 「満杯でなくなるのを待っている人」（入れたい）

`notify_one()` は**誰が起きるかを選べません。** 起こしたい相手とは違う人が起きて、
「自分の条件はまだ偽だ」と分かってまた眠ると、**通知はそこで消えます。**
本当に起きるべきだった人は、誰にも起こされないまま残ります。
これを **lost wakeup（通知の取りこぼし）** と呼びます。

`notify_all()` なら全員起こすので取りこぼしは起きず、**正しさは保てます。**
ただし「起きたけれど自分の番ではなかった」人が増えるぶん、無駄が増えます。

> **待つ理由が2種類あるなら、条件変数も2本用意する。**
> **鍵の本数は「守るデータの数」、条件変数の本数は「待つ理由の数」で決まる。**

実際のプログラムのキューを読むときは、この3点だけ確かめれば構造が分かります。

1. **鍵は何本で、何を守っているか**
2. **条件変数は何本で、それぞれ何を待っているか**
3. **どの操作が、どの条件変数を起こしているか**

## 問2-3. 待つ `pop` と待たない `pop`

`ConcurrentQueue::pop()` は**空なら待ちます。失敗しません。**
一方、実用のキューはたいてい「待たない `bool try_pop(T& out)`」も持っています。

- **待たない形が要るのは、どんな場面でしょうか**
- **待つ形が要るのは、どんな場面でしょうか**

自分で考えてから、下を見てください。

### 問2-3 の解答

**待たない形（`bool try_pop(T&)`）が要る場面**

- **表示のように「最新だけあればよい」処理。** 新しいフレームが来ていれば描き、
  来ていなければ前のフレームのまま次へ進む。ここで待つと**画面が固まります**
- **終了処理。** 「残っているものを全部吸い出して終わる」というとき、
  待つ形では最後に空になった瞬間に固まります
- **他にもやることがあるスレッド。** なければ別の仕事に移りたい場合

**待つ形（`T pop()`）が要る場面**

- **パイプラインの中段。** 前の段からデータが来るまで、やることは何もありません。
  ここで待たずに「空か？空か？」と繰り返す（**ポーリング**）と、CPU を無駄に食います
- **正しさのために「必ず1個受け取る」必要がある場合。** 失敗しないぶん、使う側が単純になります

実用のキューは、たいてい**3つ**を用意します。

- 待つ `pop()`
- 待たない `try_pop(T&)`
- **時間を切って待つ** `pop(T&, timeout)`

3つ目が実は重要で、「基本は待つが、いつまでも待ち続けはしない」という形が書けます。
**終了フラグを見に戻る隙ができる**ので、安全に終われるキューにはほぼ必ず入っています。